# Chapter 3 — Simple Exponential Smoothing (SES)
## Case Study: Cold Chain Pharma — Demand Smoothing with Level Shifts

**Reference:** Vandeput (2021), Chapter 3  
**Author:** Hanzila Bin Younus — EM CDE, University of Salzburg

---

### Industry Scenario

> **Company:** Pharmaceutical distributor managing cold chain inventory  
> **Problem:** A product line experienced a demand level shift mid-year (new hospital
> contract). The planning team needs a model that adapts to this shift **without**
> being destabilised by weekly noise. How should they tune alpha?

### SES Formula
$$f_{t+1} = \alpha \cdot d_t + (1-\alpha) \cdot f_t$$

**Alpha intuition:**
- α → 1: "I almost only trust last week. History barely matters."
- α → 0: "I trust all history equally. I change very slowly."

### Model Memory Concept (Vandeput Ch.3 insight)
The weight given to observation $d_{t-k}$ (k periods ago) is:
$$w_k = \alpha(1-\alpha)^k$$
This is exponential decay — the name comes from this property.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fb',
                     'axes.spines.top': False, 'axes.spines.right': False})
C = {'demand': '#1a2332', 'naive': '#94a3b8',
     'low': '#6246ea', 'mid': '#0077b6', 'high': '#e07b00', 'opt': '#1a7f5a'}

In [ ]:
# ── 1. PHARMA DEMAND WITH LEVEL SHIFT ─────────────────────────────────
np.random.seed(7)
n = 72  # 72 weeks (18 months)

# Phase 1: baseline demand (weeks 0-35)
# Phase 2: post-contract demand shift (weeks 36-71)
demand = np.concatenate([
    np.clip(800 + np.random.normal(0, 45, 36), 0, None),
    np.clip(1150 + np.random.normal(0, 55, 36), 0, None)
])

TRAIN_END = 54  # 75% train
EXTRA = 8

print(f'Phase 1 mean (pre-contract): {demand[:36].mean():.0f} units/week')
print(f'Phase 2 mean (post-contract): {demand[36:].mean():.0f} units/week')
print(f'Level shift magnitude: +{demand[36:].mean()-demand[:36].mean():.0f} units (+{(demand[36:].mean()/demand[:36].mean()-1)*100:.0f}%)')

In [ ]:
# ── 2. SES IMPLEMENTATION ─────────────────────────────────────────────
def ses(d, alpha=0.3, extra=8):
    """
    Simple Exponential Smoothing.
    f_{t+1} = alpha * d_t + (1-alpha) * f_t
    """
    cols = len(d)
    f = np.full(cols + extra, np.nan)
    a = np.full(cols + extra, np.nan)
    a[0] = d[0]; f[1] = a[0]
    for t in range(1, cols):
        a[t] = alpha * d[t] + (1 - alpha) * a[t-1]
        if t+1 < cols+extra: f[t+1] = a[t]
    f[cols:] = a[cols-1]
    return f, a

def compute_mae(actual, forecast, train_end):
    a = actual[train_end:]
    f = forecast[train_end:train_end+len(a)]
    mask = ~(np.isnan(a)|np.isnan(f))
    return np.abs((f-a)[mask]).mean()

# Test at multiple alpha values
alphas = [0.05, 0.20, 0.50, 0.90]
forecasts = {a: ses(demand, alpha=a, extra=EXTRA)[0] for a in alphas}

print('SES forecasts computed')
print(f'Test MAE by alpha:')
for a, f in forecasts.items():
    print(f'  α={a:.2f}: MAE={compute_mae(demand, f, TRAIN_END):.1f}')

In [ ]:
# ── 3. MODEL MEMORY VISUALISATION ─────────────────────────────────────
# Show how much weight each past period gets for different alpha values
fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, alpha, color in zip(axes, [0.05, 0.20, 0.50, 0.90],
                              [C['low'], C['mid'], C['high'], '#c0392b']):
    n_periods = 30
    weights = [(1-alpha)**i * alpha for i in range(n_periods)]
    weights.reverse()
    opacities = np.linspace(0.2, 1.0, n_periods)

    bars = ax.bar(range(-n_periods+1, 1), weights,
                  color=color, alpha=0.7, edgecolor='white', linewidth=0.3)
    for bar, op in zip(bars, opacities):
        bar.set_alpha(op)

    # Shade 80% weight region
    cum_w = np.cumsum(weights[::-1])
    idx_80 = np.searchsorted(cum_w, 0.8)
    ax.axvline(-idx_80+1, color='#c0392b', lw=1.5, ls='--', alpha=0.7)
    ax.text(-idx_80, max(weights)*0.7, f'80% weight\nin last {idx_80} periods',
            fontsize=7, color='#c0392b')

    ax.set_title(f'α = {alpha}', fontsize=11, fontweight='bold', color=color)
    ax.set_xlabel('Periods Ago', fontsize=9)
    if alpha == 0.05:
        ax.set_ylabel('Weight Assigned', fontsize=9)

fig.suptitle('Model Memory — How Much Weight Does Each Past Period Get?',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('output_ch03_model_memory.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output_ch03_model_memory.png')

In [ ]:
# ── 4. LEVEL SHIFT RESPONSE ───────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 9),
                          gridspec_kw={'height_ratios': [3,1]})
ax = axes[0]
x = np.arange(len(demand) + EXTRA)

ax.axvspan(0, TRAIN_END, alpha=0.05, color='#0077b6')
ax.axvspan(TRAIN_END, len(demand), alpha=0.05, color='#6246ea')
ax.axvline(36, color='#c0392b', lw=1.5, ls=':', alpha=0.6,
           label='Contract start (level shift)')
ax.axvline(TRAIN_END, color='#6246ea', lw=1.2, ls='--', alpha=0.5)

ax.plot(np.arange(len(demand)), demand, color=C['demand'],
        lw=2, label='Actual Demand', zorder=5)

labels = {0.05: 'α=0.05 (very slow)', 0.20: 'α=0.20 (balanced)',
          0.50: 'α=0.50 (reactive)', 0.90: 'α=0.90 (very fast)'}
colors_map = {0.05: C['low'], 0.20: C['mid'], 0.50: C['high'], 0.90: '#c0392b'}

for alpha, f in forecasts.items():
    ax.plot(x[:len(f)], f, color=colors_map[alpha], lw=1.7,
            ls='--', label=labels[alpha], alpha=0.85)

ax.annotate('New hospital\ncontract starts', xy=(36, demand[36]),
            xytext=(28, demand[36]+200), fontsize=9, color='#c0392b',
            arrowprops=dict(arrowstyle='->', color='#c0392b'))
ax.text(2, demand.max()*0.93, 'TRAIN', color='#0077b6', fontsize=9, fontweight='bold')
ax.text(TRAIN_END+1, demand.max()*0.93, 'TEST', color='#6246ea', fontsize=9, fontweight='bold')

ax.set_title('SES Response to Demand Level Shift — Pharma Cold Chain',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Weekly Demand (units)'); ax.legend(fontsize=9, loc='upper left')

# Residuals
ax2 = axes[1]
for alpha, f in forecasts.items():
    n_c = min(len(demand), len(f))
    ax2.plot(f[:n_c]-demand[:n_c], color=colors_map[alpha],
             lw=1.2, label=f'α={alpha}', alpha=0.8)
ax2.axhline(0, color='black', lw=0.8, ls='--')
ax2.axvline(36, color='#c0392b', lw=1, ls=':', alpha=0.5)
ax2.axvline(TRAIN_END, color='#6246ea', lw=1.2, ls='--', alpha=0.5)
ax2.set_ylabel('Error'); ax2.set_xlabel('Week'); ax2.legend(fontsize=8)
ax2.set_title('Residuals — notice how fast each α adapts to the level shift', fontsize=9)

plt.tight_layout()
plt.savefig('output_ch03_ses_level_shift.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output_ch03_ses_level_shift.png')

In [ ]:
# ── 5. ALPHA OPTIMISATION ─────────────────────────────────────────────
# Grid search for optimal alpha on training data
alpha_range = np.arange(0.01, 1.00, 0.01)
mae_by_alpha = []

for a in alpha_range:
    f, _ = ses(demand[:TRAIN_END], alpha=a, extra=0)
    mask  = ~np.isnan(f[:TRAIN_END])
    mae   = np.abs((f[:TRAIN_END] - demand[:TRAIN_END])[mask]).mean()
    mae_by_alpha.append(mae)

opt_alpha = alpha_range[np.argmin(mae_by_alpha)]
opt_mae   = min(mae_by_alpha)

# Compare on test set
f_opt, _ = ses(demand, alpha=opt_alpha, extra=EXTRA)
test_mae_opt = compute_mae(demand, f_opt, TRAIN_END)

# Plot MAE landscape
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(alpha_range, mae_by_alpha, color=C['mid'], lw=2)
ax.axvline(opt_alpha, color=C['opt'], lw=2, ls='--',
           label=f'Optimal α={opt_alpha:.2f} (train MAE={opt_mae:.0f})')
ax.fill_between(alpha_range, mae_by_alpha,
                min(mae_by_alpha) + 30, alpha=0.08, color=C['mid'])
for a, name, col in [(0.05,'very slow',C['low']),(0.90,'very fast','#c0392b')]:
    idx = np.argmin(np.abs(alpha_range - a))
    ax.scatter([a], [mae_by_alpha[idx]], color=col, s=60, zorder=5)
    ax.annotate(name, (a, mae_by_alpha[idx]), xytext=(a+0.03, mae_by_alpha[idx]+8),
                fontsize=8, color=col)
ax.set_xlabel('Alpha (α)', fontsize=10)
ax.set_ylabel('Training MAE (units)', fontsize=10)
ax.set_title('Alpha Optimisation — MAE Landscape (Training Set)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('output_ch03_alpha_optimisation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nOptimal alpha (train): {opt_alpha:.2f}')
print(f'Training MAE:   {opt_mae:.0f}')
print(f'Test MAE:       {test_mae_opt:.0f}')
print(f'\nVandeput rule: if optimal α > 0.60, something is wrong with the model.')
print(f'Our optimal is {opt_alpha:.2f} — within the acceptable range.')
print('Saved: output_ch03_alpha_optimisation.png')

## Summary

| Alpha | Adaptation Speed | Risk | Best For |
|-------|-----------------|------|----------|
| 0.05 | Very slow | Misses real level shifts | Very stable demand |
| 0.20 | Moderate | Balanced | Most FMCG products |
| 0.50 | Fast | Sensitive to noise | Volatile demand |
| 0.90 | Very fast | Chases random variation | Real-time event response only |

**Pharma case answer:** α=0.25–0.35 balances noise rejection with fast adaptation  
to the contract-driven level shift. α=0.05 would take 10+ weeks to detect the shift.  

**Next:** `04_double_exponential_smoothing.ipynb` — adds trend component for growing demand